Importer les dépendances

In [ ]:
import pandas as pd
import numpy as np
import pickle

Charger et Explorer les données

In [ ]:
# Chargement du fichier pickle
with open('train_data.pkl', 'rb') as f:
    train_data = pickle.load(f)

images = train_data['images']
labels = train_data['labels']

print("------------Analyse data d'entraînement:-------------")

# Print des formes des images et labels
print(f"Nombre total d'image: {len(images)}")
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

# Print du type des labels
print("Labels type: ", type(labels[0]))
# Résultat: Chaque label est un array de dimension 1

# Print des valeurs manquantes
print("Missing values in images:", np.isnan(images).sum().sum())
print("Missing values in labels:", np.isnan(labels).sum())
# Résultat: Aucune valeur manquante

# Print des valeurs min et max des labels
print("Min label:", labels.min())
print("Max label:", labels.max())
# Résultat: Chaque classe est entre 0 et 4

# Print du nombre de données dans chaque classe
classes, counts = np.unique(labels, return_counts=True)
for c, n in zip(classes, counts):
    print(f"Class {c}: {n}")
# Résultat: Il y a beaucoup de classe 0 et peu de classe 4

# Analyser les statistiques des 3 couleurs des canaux
print(f"\n----------------L'importance de chaque canal----------------")
channels = ['Rouge', 'Vert', 'Bleu']

for i, channel in enumerate(channels):
    channel_data = images[:, :, :, i]
    print(f"Canal {channel}:")
    print(f"  Min: {channel_data.min()}, Max: {channel_data.max()}")
    print(f"  Moyenne: {channel_data.mean():.2f}, Écart-type: {channel_data.std():.2f}")

print("\n---------------- Importance des canaux par CLASSE ----------------")

unique_labels = np.unique(labels)

for class_label in unique_labels:
    print(f"\n-- Classe {class_label} --")

    # Extraire les images de cette classe
    class_indices = np.where(labels == class_label)[0]
    class_images = images[class_indices]

    for i, channel in enumerate(channels):
        channel_data = class_images[:, :, :, i]

        print(f"  Canal {channel}:")
        print(f"    Min: {channel_data.min():.2f}, Max: {channel_data.max():.2f}")
        print(f"    Moyenne: {channel_data.mean():.2f}")
        print(f"    Écart-type: {channel_data.std():.2f}")

# Remarque: Les moyennes de chaque couleurs diffèrent selon la classe (surtout
# pour les couleurs rouges et bleues). Il n'y a presque pas de vert dans les
# images.

print(f"\n---------------Luminosité des images---------------")
luminosites = []
for img in images:
    # Moyenne des 3 canaux = luminosité
    gris = np.mean(img, axis=2)
    luminosite_moyenne = np.mean(gris)
    luminosites.append(luminosite_moyenne)

luminosites = np.array(luminosites)

print(f"Nombre d'images analysées: {len(luminosites)}")
print(f"Luminosité moyenne: {np.mean(luminosites):.1f}")
print(f"Luminosité min: {np.min(luminosites):.1f}")
print(f"Luminosité max: {np.max(luminosites):.1f}")

print(f"\n---------------Luminosité des images par classe---------------")

for class_label in unique_labels:
    print(f"\n-- Classe {class_label} --")

    # Extraire les images de cette classe
    class_indices = np.where(labels == class_label)[0]
    class_images = images[class_indices]

    luminosites = []
    for img in class_images:
        # Moyenne des 3 canaux = luminosité
        gris = np.mean(img, axis=2)
        luminosite_moyenne = np.mean(gris)
        luminosites.append(luminosite_moyenne)

    luminosites = np.array(luminosites)

    print(f"Nombre d'images analysées: {len(luminosites)}")
    print(f"Luminosité moyenne: {np.mean(luminosites):.1f}")
    print(f"Luminosité min: {np.min(luminosites):.1f}")
    print(f"Luminosité max: {np.max(luminosites):.1f}")

  # Remarque: La luminosité semble descendre plus la classe augmente.


Prétraitement des données

In [ ]:
def data_processor(images):
    n_images = images.shape[0]

    # h: height, w: width, c: color
    h, w, c = images.shape[1], images.shape[2], images.shape[3]

    # Traiter chaque canal couleur séparément
    X_processed = []  # Données de chaque canal
    for canal in range(3):  # R, G, B
      # Aplatissage
      canal_data = images[:, :, :, canal].reshape(n_images, -1)
      # Mise à l'échelle des données entre 0 et 1
      canal_data = canal_data.astype("float32") / 255.0

      # Normalisation z-score canal par canal
      mean = canal_data.mean(axis=0)
      std = canal_data.std(axis=0) + 1e-8
      canal_data = (canal_data - mean) / std

      X_processed.append(canal_data)

      # Concaténer les canaux
      X = np.concatenate(X_processed, axis=1)

    return X

# Nouvelles données
processed_images = data_processor(images)

# On sort les labels de leur array
int_labels = labels.reshape(-1)

print(f"Forme version par couleur: {processed_images.shape}")

Préparer les données d'entraînement et de validation

In [ ]:
np.random.seed(0)
indices = np.random.permutation(len(labels))


print("X_TRAIN SHAPE: ", X_train.shape)
print("Y_TRAIN SHAPE: ", y_train.shape)
print(y_train)
print("Labels min:", y_train.min())
print("Labels max:", y_train.max())
print("Unique labels:", np.unique(y_train)[:20])

Entraîner les modèles

In [ ]:
def minkowski_mat(x, Y, p=2):
    return (np.sum((np.abs(x - Y)) ** p, axis=1)) ** (1.0 / p)

class NeighborhoodClassifier:
    def __init__(self, parzen=False, dist_func=minkowski_mat, k=1, radius=0.4):
        self.parzen = parzen
        self.dist_func = dist_func
        self.k = k
        self.radius = radius

    # The train function for knn / Parzen windows is really only storing the
    # dataset
    def train(self, train_inputs, train_labels):
        self.train_inputs = train_inputs
        self.train_labels = train_labels.flatten() if train_labels.ndim > 1 else train_labels
        # Correctly determine the number of classes based on the maximum label
        # value
        self.n_classes = int(np.max(train_labels)) + 1  # Class starting to 0

    # The prediction function takes as input test_data and returns an array
    # containing the predicted classes.
    def compute_predictions(self, test_data):
        # Initialization of the count matrix and the predicted classes array
        num_test = test_data.shape[0]
        counts = np.zeros((num_test, self.n_classes))
        classes_pred = np.zeros(num_test)

        # For each test datapoint
        for (i, ex) in enumerate(test_data):

            # i is the row index
            # ex is the i'th row

            # Find the distances to each training set point using dist_func
            distances = self.dist_func(ex, self.train_inputs)
            M = len(distances)

            # Go through the training set to find the neighbors of the current
            # point (ex)
            # You will distinguish between Parzen and KNN here
            ind_neighbors= []
            if self.parzen:
                radius = self.radius
                while len(ind_neighbors) == 0:
                    ind_neighbors = np.array([j for j in range(M)
                    if distances[j] < radius])
                    radius *= 2
            else:
                ind_neighbors = np.argsort(distances)[:self.k]
            # Calculate the number of neighbors belonging to each class and
            # write them in counts[i,:]
            cl_neighbors = list(self.train_labels[ind_neighbors])
            for j in range(min(len(cl_neighbors) if self.parzen else self.k,
                               self.train_inputs.shape[0])):
                counts[i, int(cl_neighbors[j])] += 1

            # From the counts matrix, define classes_pred[i] (don't forget that
            # classes are labeled from 1 to n)
            classes_pred[i] = np.argmax(counts[i, :])


        return classes_pred

def conf_matrix_vectorized(testlabels, predlabels):
    testlabels = testlabels.flatten() if testlabels.ndim > 1 else testlabels
    predlabels = predlabels.flatten() if predlabels.ndim > 1 else predlabels

    n_classes = max(testlabels.max(), predlabels.max()) + 1
    matrix = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(matrix, (testlabels.astype(int), predlabels.astype(int)), 1)
    return matrix

def get_test_error(k_or_radius, X_train, y_train, X_test, y_test, parzen=False):
  if parzen:
        classifier = NeighborhoodClassifier(parzen=True, dist_func=minkowski_mat,
                                            radius=k_or_radius)
  else:
        classifier = NeighborhoodClassifier(parzen=False, dist_func=minkowski_mat,
                                            k=k_or_radius)

  classifier.train(X_train, y_train)
  classes_pred = classifier.compute_predictions(X_test)

  confmat = conf_matrix_vectorized(y_test, classes_pred)
  sum_correct = np.sum(np.diag(confmat))
  sum_preds = np.sum(confmat)

  return 1.0 - sum_correct / sum_preds

# trouver le meilleur k
error_k = [get_test_error(k, X_train, y_train, X_valid,
                          y_valid, parzen=False) for k in range(1, 27)]
chosen_k = np.argmin(error_k) + 1
print(f"Le k choisis: {chosen_k}")
error_knn = get_test_error(chosen_k, X_train, y_train, X_valid, y_valid, parzen=False)
accuracy_knn = 1 - error_knn
print(f"Accuracy du modèle KNN: {accuracy_knn:.4f}")

# trouver le meilleur radius
radius_values = np.arange(0.1, 10.1, 0.5)
error_radius = [get_test_error(radius, X_train, y_train,
                               X_valid, y_valid, parzen=True)
                               for radius in radius_values]
chosen_radius = radius_values[np.argmin(error_radius)]
print(f"Le radius choisis: {chosen_radius}")
error_parzen = get_test_error(chosen_radius, X_train, y_train, X_valid, y_valid, parzen=True)
accuracy_parzen = 1 - error_parzen
print(f"Accuracy du modèle Parzen: {accuracy_parzen:.4f}")



# Number of neighbors (k) for knn
k = chosen_k
radius = chosen_radius

# Create the classifiers
knn = NeighborhoodClassifier(parzen=False, dist_func=minkowski_mat, k=k)
parzen = NeighborhoodClassifier(parzen=True, dist_func=minkowski_mat,
                                radius=radius)

# We train the models
knn.train(X_train, y_train)
parzen.train(X_train, y_train)

# Classificateur linéaire (classe parent)
class LinearModel:
  def __init__(self, w0, reg):
    self.w = np.array(w0, dtype=float)
    self.reg = reg

  def predict(self, X):
      """Retourne f(x) pour un batch X
      """
      return np.dot(X, self.w)

  def error_rate(self, X, y):
      return np.mean(np.sign(self.predict(X)) != y)

  # les méthodes loss et gradient seront redéfinies dans les classes enfants
  def loss(self, X, y):
      return 0

  def gradient(self, X, y):
      return self.w

  def train(self, X, y, stepsize, n_steps, plot=False):
      """Faire la descente du gradient avec batch complet pour n_steps itération
      et un taux d'apprentissage fixe. Retourne les tableaux de loss et de
      taux d'erreur vu apres chaque iteration.
      """
      losses = []
      errors = []

      for i in range(n_steps):
          # Gradient Descent
          self.w -= stepsize * self.gradient(X, y)

          # Update losses
          losses += [self.loss(X, y)]

          # Update errors
          errors += [self.error_rate(X, y)]

      print("Training completed: the train error is {:.2f}%".format(errors[-1]*100))
      return np.array(losses), np.array(errors)

class LogisticRegression(LinearModel):

    def __init__(self, w0, reg):
        super().__init__(w0, reg)

    def loss(self, X, y):
        return np.mean(np.log(1 + np.exp(-y * self.predict(X)))) + .5 * self.reg * np.sum(self.w ** 2)

    def gradient(self, X, y):
        # Régularisation L2
        probas = 1 / (1 + np.exp(y * self.predict(X)))
        return ((probas * -y)[:, np.newaxis] * X).mean(axis=0) + self.reg * self.w

# Fonction qui s'occupe d'entrainer le modèle pour chaque classe vs le reste
# des classes
def ovr_train(X, y, modelclass, reg, stepsize, n_steps):
    classes = np.unique(y)
    models = []

    # Un modèle différent pour séparer chaque classe des autres
    # Probleme multiclasse à binaire
    for c in classes:
        y_bin = np.where(y == c, 1, -1)
        w0 = np.zeros(X.shape[1])
        model = modelclass(w0, reg)
        model.train(X, y_bin, stepsize, n_steps)
        models.append(model)

    return classes, models  # Un modèle avec poids différents pour chaque classe


def ovr_predict(X, classes, models):
    scores = np.column_stack([m.predict(X) for m in models])
    # On garde la classe avec le meilleur score pour chaque ligne de X
    return classes[np.argmax(scores, axis=1)]

def add_bias(X):
  return np.hstack([np.ones((X.shape[0],1)), X])

# On ajoute une colonne de 1 devant les features pour permettre au modèle
# d'apprendre un billet
X_train_b = add_bias(X_train)
X_valid_b = add_bias(X_valid)
classes, models = ovr_train(
    X_train_b, y_train,
    LogisticRegression, reg=0.01, stepsize=0.1, n_steps=200
)

y_pred = ovr_predict(X_valid_b, classes, models)

print("Accuracy:", np.mean(y_pred == y_valid))



Évaluer les modèles



In [ ]:
with open('test_data.pkl', 'rb') as f:
    test_data = pickle.load(f)

test_images = test_data['images']

# Prétraitement des données de test
X_test = data_processor(test_images)

# Ajouter le biais comme au training
X_test_b = add_bias(X_test)


prediction = ovr_predict(X_test_b, classes, models)

print("Nombre de prédictions:", len(prediction))



Générer les prédictions pour la soumission Kaggle

In [ ]:
import csv

preds = prediction.astype(int)
print(preds)

with open("prediction.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ID", "prediction"])

    for idx, p in enumerate(preds):
        writer.writerow([idx + 1, p])
